# M03 — Stacking, Blending & Ensemble Methods

## Why ensembles work: the bias-variance decomposition

The expected prediction error decomposes as:
$$\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

**Bagging** (Random Forest, BaggingClassifier) reduces **variance** — average many high-variance models.
**Boosting** (XGBoost, LightGBM) reduces **bias** — sequentially correct previous errors.
**Stacking** reduces **both** — a meta-learner learns when to trust each base model.

## Stacking vs Blending

**Stacking:**
1. Split train into $k$ folds
2. For each fold: train base models on $k-1$ folds, predict the held-out fold → out-of-fold (OOF) predictions
3. OOF predictions become features for the meta-model
4. Train meta-model on OOF predictions
5. Final prediction = meta-model(base_model_predictions on test)

**Blending:**
- Simpler: use a fixed holdout set (not CV) to train the meta-model
- Less data-efficient but faster

**Key rule:** Base models in the stack must be diverse. If all base models make the same errors, the meta-model can't fix them.

**Reference:** [sklearn Stacking](https://scikit-learn.org/stable/modules/ensemble.html)


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.datasets import fetch_openml, fetch_california_housing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier, VotingClassifier, BaggingClassifier
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')
credit['target'] = (credit['class'] == 'good').astype(int)

cat_cols = credit.select_dtypes(include='object').columns.drop('class').tolist()
num_cols = credit.select_dtypes(include='number').columns.drop('target').tolist()

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
])
X = credit[num_cols + cat_cols]
y = credit['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_tr_enc = preprocessor.fit_transform(X_train)
X_te_enc = preprocessor.transform(X_test)

print(f"Ready. Features: {X_tr_enc.shape[1]}")

---
## Exercise 1 — Voting Ensemble: Soft vs Hard

**Task:** Build a voting ensemble and understand the difference between soft and hard voting.

**Hard voting:** majority class wins. Each classifier votes for one class.
**Soft voting:** average predicted probabilities. Uses the confidence of each classifier.

1. Build 3 base classifiers inside preprocessing pipelines: LogisticRegression, RandomForest, LGBMClassifier.
2. Create `VotingClassifier` with `voting='soft'` and `voting='hard'`.
3. Evaluate both on test set: accuracy, AUC (soft only), F1.
4. Compare to each base model individually.
5. Return `voting_comparison` DataFrame.

In [ ]:
# YOUR CODE HERE
voting_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert voting_comparison is not None
assert len(voting_comparison) >= 5  # 3 base + 2 ensembles
model_names = voting_comparison.index.tolist() if voting_comparison.index.name else voting_comparison.iloc[:,0].tolist()
assert any('soft' in str(n).lower() for n in model_names)
assert any('hard' in str(n).lower() for n in model_names)
print("✓ Exercise 1 passed")
print(voting_comparison.to_string())

---
## Exercise 2 — Out-of-Fold Stacking from Scratch

**Task:** Implement stacking manually — this teaches the concept far better than using sklearn's API.

1. Implement `generate_oof_predictions(models, X_train, y_train, n_folds=5)` that:
   - For each model: run $k$-fold CV, collect out-of-fold predictions (probabilities)
   - Return `oof_matrix`: shape `(n_train, n_models)` — each column = OOF preds for one model
   - Also return `test_preds_matrix`: shape `(n_test, n_models)` — each column = mean prediction across all $k$ models trained on each fold

2. Train a LogisticRegression meta-model on `oof_matrix` with `y_train`.
3. Make final predictions: `meta_model.predict_proba(test_preds_matrix)[:, 1]`.
4. Compare stacked AUC vs individual base model AUCs.
5. Return `stacking_results` DataFrame.

In [ ]:
from sklearn.base import clone

def generate_oof_predictions(models: list, X_train, y_train, X_test, n_folds: int = 5):
    """
    Generate out-of-fold predictions for stacking.
    Returns (oof_matrix, test_preds_matrix)
    """
    # YOUR CODE HERE
    pass

base_models = [
    LogisticRegression(max_iter=1000, random_state=42),
    RandomForestClassifier(n_estimators=100, random_state=42),
    lgb.LGBMClassifier(random_state=42, verbosity=-1)
]

# YOUR CODE HERE
stacking_results = None

In [ ]:
# --- ASSERTIONS ---
assert stacking_results is not None
assert 'stacked' in str(stacking_results).lower()
auc_col = [c for c in stacking_results.columns if 'auc' in c.lower()][0]
stacked_auc = stacking_results.loc[
    stacking_results.iloc[:,0].str.lower().str.contains('stack'), auc_col
].values[0]
base_aucs = stacking_results.loc[
    ~stacking_results.iloc[:,0].str.lower().str.contains('stack'), auc_col
].values
assert stacked_auc >= base_aucs.min(), "Stacking should at least match worst base model"
print(f"✓ Exercise 2 passed — Stacked AUC: {stacked_auc:.4f}")
print(stacking_results.to_string(index=False))

---
## Exercise 3 — sklearn StackingClassifier

**Task:** Now use sklearn's `StackingClassifier` — faster to write but same concept.

1. Build a 2-level stack:
   - Level 1 (base estimators): LR, RF, XGBoost, LightGBM — each inside their own preprocessing pipeline
   - Level 2 (meta-learner): `LogisticRegression(C=0.1)` trained on the OOF outputs
2. Use `passthrough=True` to also include original features at the meta level.
3. Evaluate with 5-fold CV: `cross_val_score(..., scoring='roc_auc')`.
4. Compare to the best single model from previous exercises.
5. Inspect `stack.estimators_` — verify all 4 base models are fitted.

In [ ]:
# YOUR CODE HERE
stack = None
stack_cv_scores = None

In [ ]:
# --- ASSERTIONS ---
assert isinstance(stack, StackingClassifier)
stack.fit(X_tr_enc, y_train)
assert len(stack.estimators_) == 4
assert stack_cv_scores is not None and len(stack_cv_scores) == 5
assert stack_cv_scores.mean() > 0.65
print(f"✓ Exercise 3 passed — Stack CV AUC: {stack_cv_scores.mean():.4f} ± {stack_cv_scores.std():.4f}")

---
## Exercise 4 — Diversity Measure: Error Correlation

**Concept:** Ensembles work best when base models are **diverse** — they make different errors. If all models fail on the same samples, the ensemble can't fix it.

**Measuring diversity:** compute the correlation matrix of base model errors. Low correlation = high diversity = good ensemble candidate.

1. Get OOF predictions from Exercise 2.
2. Compute per-sample error for each model: `|y_pred_oof - y_train|`.
3. Build `error_correlation`: Pearson correlation matrix of error vectors.
4. Identify the **most diverse pair** (lowest correlation).
5. Train a 2-model ensemble with only the most diverse pair and compare to the full 3-model ensemble.

In [ ]:
# YOUR CODE HERE
error_correlation = None
most_diverse_pair = None  # tuple of (model_name_1, model_name_2)

In [ ]:
# --- ASSERTIONS ---
assert error_correlation is not None
assert error_correlation.shape == (3, 3)
assert np.allclose(np.diag(error_correlation.values), 1.0), "Diagonal must be 1"
assert most_diverse_pair is not None and len(most_diverse_pair) == 2
print("✓ Exercise 4 passed")
print("Error correlation matrix:")
print(error_correlation.round(3).to_string())
print(f"Most diverse pair: {most_diverse_pair}")

---
## Exercise 5 — Weighted Average Blending

**Task:** Find optimal ensemble weights using hill-climbing optimization.

Given test predictions from 4 models (LR, RF, XGBoost, LightGBM), find the weights $w_1, w_2, w_3, w_4$ (summing to 1) that maximize AUC on a validation set.

1. Implement `optimize_blend_weights(val_preds_matrix, y_val, n_iters=1000)` using random search:
   - Sample random weight vectors from Dirichlet distribution
   - Keep the weights that maximize AUC
2. Apply optimal weights to test set.
3. Compare blended AUC vs equal-weight average vs best single model.
4. Return `blend_results` and `optimal_weights`.

In [ ]:
def optimize_blend_weights(val_preds: np.ndarray, y_val: np.ndarray,
                            n_iters: int = 1000, random_state: int = 42) -> np.ndarray:
    """
    Find blend weights maximizing AUC via random search.
    val_preds: (n_samples, n_models)
    Returns optimal_weights: (n_models,) summing to 1.
    """
    # YOUR CODE HERE
    # Hint: np.random.dirichlet([1,1,1,1]) generates random weights summing to 1
    pass

# YOUR CODE HERE
blend_results = None
optimal_weights = None

In [ ]:
# --- ASSERTIONS ---
assert optimal_weights is not None
assert abs(sum(optimal_weights) - 1.0) < 1e-6, "Weights must sum to 1"
assert all(w >= 0 for w in optimal_weights), "All weights must be non-negative"
print(f"✓ Exercise 5 passed")
print(f"Optimal weights: {[f'{w:.3f}' for w in optimal_weights]}")
if blend_results is not None:
    print(blend_results)

---
## Exercises 6–10: Remaining Ensemble Topics

**Exercise 6 — BaggingClassifier: Variance Reduction**
Train `BaggingClassifier` with `base_estimator=DecisionTreeClassifier(max_depth=None)`. Vary `n_estimators` (1, 5, 10, 25, 50, 100). Show how test variance decreases as n_estimators grows. Implement bootstrap sampling from scratch and verify it matches sklearn's implementation.

**Exercise 7 — AdaBoost from Scratch**
Implement AdaBoost with decision stumps (max_depth=1) from scratch using numpy. Each round: (1) train weak learner on weighted samples, (2) compute weighted error, (3) compute alpha = 0.5 * log((1-err)/err), (4) update sample weights. Compare to sklearn AdaBoostClassifier.

**Exercise 8 — Multi-Level Stacking**
Build a 3-level stack: Level 1 = 4 diverse models, Level 2 = 2 meta-models trained on OOF from Level 1, Level 3 = final blender. Assess whether the extra level helps or hurts (overfitting risk).

**Exercise 9 — Ensemble Pruning**
Start with 10 models. Implement greedy ensemble pruning: iteratively add the model that most improves the ensemble AUC, stop when adding more models doesn't help. Return the optimal subset size and which models were selected.

**Exercise 10 — Capstone: Competition-Style Ensemble**
Build the best possible model on the credit dataset using any combination of: feature engineering, XGBoost, LightGBM, stacking, blending, threshold tuning, calibration. Target: beat 0.78 AUC. Document every decision with a markdown cell explaining why you made that choice.

In [ ]:
# Exercise 6: BaggingClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# YOUR CODE HERE for Exercise 6
bagging_results = None  # DataFrame: n_estimators, train_auc, test_auc, variance

In [ ]:
# Exercise 7: AdaBoost from scratch
def adaboost_from_scratch(X_train, y_train, X_test, n_rounds=50):
    """
    AdaBoost with decision stumps.
    Returns test predictions.
    """
    # YOUR CODE HERE
    pass

# Verify against sklearn
scratch_preds = adaboost_from_scratch(X_tr_enc, y_train.values, X_te_enc)
sklearn_ada = AdaBoostClassifier(n_estimators=50, random_state=42)
sklearn_ada.fit(X_tr_enc, y_train)
sklearn_preds = sklearn_ada.predict_proba(X_te_enc)[:,1]

if scratch_preds is not None:
    scratch_auc = roc_auc_score(y_test, scratch_preds)
    sklearn_auc = roc_auc_score(y_test, sklearn_preds)
    print(f"Scratch AdaBoost AUC: {scratch_auc:.4f} | sklearn: {sklearn_auc:.4f}")
    assert abs(scratch_auc - sklearn_auc) < 0.05, "Should be within 5% of sklearn"
    print("✓ Exercise 7 passed")

In [ ]:
# Exercise 10: Capstone
# YOUR CODE HERE
final_model = None
final_auc = None

if final_auc is not None:
    assert final_auc >= 0.75, f"Target AUC >= 0.75, got {final_auc:.4f}"
    print(f"✓ Exercise 10 passed — Final AUC: {final_auc:.4f}")